In [6]:

import pandas as pd


# STAGE 1: DATA LOADING


dataset = pd.read_csv("Data.csv")

cleaned_data = []
seen_rows = set()

missing_count = 0
invalid_count = 0
duplicate_count = 0

print("Original records:", len(dataset))



# STAGE 2: DATA CLEANING


required = ["failures", "studytime", "absences", "G3"]

for _, student in dataset.iterrows():

    # Check missing values
    if student[required].isna().any():
        missing_count += 1
        continue

    failures = student["failures"]
    studytime = student["studytime"]
    absences = student["absences"]
    G3 = student["G3"]

    # Check invalid values
    values = pd.to_numeric(
        student[required], errors="coerce"
    )

    if values.isna().any():
        invalid_count += 1
        continue

    failures = values["failures"]
    studytime = values["studytime"]
    absences = values["absences"]
    G3 = values["G3"]

    if (
        failures not in [0, 1, 2, 3]
        or studytime not in [1, 2, 3, 4]
        or absences < 0
        or not float(absences).is_integer()
        or not 0 <= G3 <= 20
    ):
        invalid_count += 1
        continue

    # Check possible duplicate records
    row_key = tuple(student.fillna("<MISSING>").tolist())

    if row_key in seen_rows:
        duplicate_count += 1
        continue

    seen_rows.add(row_key)
    cleaned_data.append(student)


# STAGE 3: INITIALISE COUNTERS


low_count = 0
medium_count = 0
high_count = 0

TP = 0
FP = 0
FN = 0
TN = 0

results = []



# STAGE 4: RISK SCORING


for student in cleaned_data:

    failures = float(student["failures"])
    studytime = float(student["studytime"])
    absences = float(student["absences"])
    G3 = float(student["G3"])

    risk_score = 0

    # Previous academic failures
    if failures >= 2:
        risk_score += 2
    elif failures == 1:
        risk_score += 1

    # Study time
    if studytime == 1:
        risk_score += 1

    # Absences
    if absences > 10:
        risk_score += 1

 
    # STAGE 5: RISK CLASSIFICATION

    if risk_score >= 3:
        risk = "High Risk"
        high_count += 1

    elif risk_score >= 1:
        risk = "Medium Risk"
        medium_count += 1

    else:
        risk = "Low Risk"
        low_count += 1

    # STAGE 6: EVALUATION USING G3


    predicted_failure = (risk == "High Risk")
    actual_failure = (G3 < 10)

    if predicted_failure and actual_failure:
        TP += 1

    elif predicted_failure and not actual_failure:
        FP += 1

    elif not predicted_failure and actual_failure:
        FN += 1

    else:
        TN += 1

    results.append({
        "failures": int(failures),
        "studytime": int(studytime),
        "absences": int(absences),
        "G3": G3,
        "risk_score": risk_score,
        "risk_category": risk,
        "actual_result": (
            "Fail" if actual_failure else "Pass"
        )
    })


# STAGE 7: FINAL RESULTS


total = TP + FP + FN + TN

if total > 0:
    accuracy = ((TP + TN) / total) * 100
else:
    accuracy = None

print("\nDATA CLEANING RESULTS")
print("Missing records:", missing_count)
print("Invalid records:", invalid_count)
print("Possible duplicates:", duplicate_count)
print("Cleaned records:", len(cleaned_data))

print("\nRISK CLASSIFICATION RESULTS")
print("Low Risk:", low_count)
print("Medium Risk:", medium_count)
print("High Risk:", high_count)

print("\nEVALUATION RESULTS")
print("True Positives:", TP)
print("False Positives:", FP)
print("False Negatives:", FN)
print("True Negatives:", TN)

if accuracy is not None:
    print(f"Prediction Accuracy: {accuracy:.2f}%")
else:
    print("Prediction Accuracy: Not available")

# Verify that all cleaned students were processed
classified_total = low_count + medium_count + high_count

print("\nVERIFICATION")
print("Total classified:", classified_total)
print("Total evaluated:", total)
print("Total cleaned:", len(cleaned_data))

assert classified_total == len(cleaned_data)
assert total == len(cleaned_data)

# Display first 10 student results
results_df = pd.DataFrame(results)

print("\nFIRST 649 STUDENT RESULTS")
print(results_df)


Original records: 649

DATA CLEANING RESULTS
Missing records: 0
Invalid records: 0
Possible duplicates: 0
Cleaned records: 649

RISK CLASSIFICATION RESULTS
Low Risk: 367
Medium Risk: 260
High Risk: 22

EVALUATION RESULTS
True Positives: 12
False Positives: 10
False Negatives: 88
True Negatives: 539
Prediction Accuracy: 84.90%

VERIFICATION
Total classified: 649
Total evaluated: 649
Total cleaned: 649

FIRST 649 STUDENT RESULTS
     failures  studytime  absences    G3  risk_score risk_category  \
0           0          2         4  11.0           0      Low Risk   
1           0          2         2  11.0           0      Low Risk   
2           0          2         6  12.0           0      Low Risk   
3           0          3         0  14.0           0      Low Risk   
4           0          2         0  13.0           0      Low Risk   
..        ...        ...       ...   ...         ...           ...   
644         1          3         4  10.0           1   Medium Risk   
645      